In [1]:
!pip install -q -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 9.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import json
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml

from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [4]:
DATASET_ROOT = Path(
    "/content/datasets"
)

DATA_YAML_PATH = Path("/content/drive/MyDrive/vision_unit_02_outputs/block_02/crack_seg_local.yaml")

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "vision_unit_02_outputs/"
    "block_04"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_NAME = "yolo26n_crack_seg_v1"
RUN_DIRECTORY = OUTPUT_ROOT / RUN_NAME


BEST_MODEL_PATH = (
    RUN_DIRECTORY
    / "weights"
    / "best.pt"
)

LAST_MODEL_PATH = (
    RUN_DIRECTORY
    / "weights"
    / "last.pt"
)

print("Dataset YAML:", DATA_YAML_PATH)
print("Output:", OUTPUT_ROOT)

Dataset YAML: /content/drive/MyDrive/vision_unit_02_outputs/block_02/crack_seg_local.yaml
Output: /content/drive/MyDrive/vision_unit_02_outputs/block_04


**Verify dataset configuration**

In [5]:
with DATA_YAML_PATH.open("r", encoding="utf-8") as file:
    dataset_config = yaml.safe_load(file)

print(yaml.safe_dump(dataset_config,sort_keys=False))

path: /content/datasets
train: images/train
val: images/val
test: images/test
names:
  0: crack



**Confirm pairing counts**

In [8]:
for split_name, expected_count in {
    "train": 3717,
    "val": 200,
    "test": 112,
}.items():
    image_directory = DATASET_ROOT / "images" / split_name
    label_directory = DATASET_ROOT / "labels" / split_name
    
    image_count = len(list(
        image_directory.glob("*.*")
    ))
    label_count = len(list(
        label_directory.glob("*.txt")
    ))
    
    print(split_name, "| images:", image_count, "| labels:", label_count)

    assert image_count == expected_count
    assert label_count == expected_count

train | images: 3717 | labels: 3717
val | images: 200 | labels: 200
test | images: 112 | labels: 112


**Load the segmentation model**

In [10]:
model = YOLO("yolo26n-seg.pt")
print("Task : ", model.task)

model.info()

Task :  segment
YOLO26n-seg summary: 309 layers, 3,126,280 parameters, 0 gradients, 10.6 GFLOPs


(309, 3126280, 0, 10.6303744)